# Module 6: Automating everyday tasks (solutions)

Python for Data: Clean, Analyse and Visualise · Avro Academy

Combine many files and write reports.

In [ ]:
# Run this cell first.
# In Colab: upload Python-for-Data-Starter-Files.zip to the Files panel (the zip itself), then run this.
# On your own computer it just checks the files are there.
import os, zipfile
from pathlib import Path
for z in sorted(Path(".").glob("Python-for-Data-Starter-Files*.zip")):
    if not Path("Python-for-Data-Starter-Files").exists():
        zipfile.ZipFile(z).extractall()
if Path("Python-for-Data-Starter-Files").is_dir():
    os.chdir("Python-for-Data-Starter-Files")
if not Path("data").is_dir() and Path("../data").is_dir():
    os.chdir("..")
needed = ["data/cafe_sales_2025.csv", "data/cafe_sales_2025_clean.csv", "data/products.csv", "data/rosters/roster_week_01.csv"]
missing = [f for f in needed if not Path(f).exists()]
print("All files found!" if not missing else f"Missing: {missing}. Upload the zip, wait for it to finish, and run this cell again.")

### Exercise 1: combine the rosters
Combine every file in `data/rosters`, add a `week` column, and show the total hours for each week.

In [ ]:
import pandas as pd
from pathlib import Path
frames = []
for f in sorted(Path("data/rosters").glob("roster_week_*.csv")):
    week = pd.read_csv(f)
    week["week"] = int(f.stem.split("_")[-1])
    frames.append(week)
roster = pd.concat(frames, ignore_index=True)
print(roster.groupby("week")["hours"].sum())

### Project: weekly roster hours report
The function from the lesson (also in `scripts/roster_report.py`).

In [ ]:
import pandas as pd
from pathlib import Path

RATES = {"Saturday": 1.25, "Sunday": 1.5}
BASE = 25

def roster_report(folder):
    files = sorted(Path(folder).glob("roster_week_*.csv"))
    roster = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    roster["weekend"] = roster["day"].isin(["Saturday", "Sunday"])
    roster["pay"] = roster["hours"] * BASE * roster["day"].map(RATES).fillna(1)
    report = roster.groupby("name").agg(
        shifts=("day", "count"),
        hours=("hours", "sum"),
        weekend_hours=("hours", lambda h: h[roster.loc[h.index, "weekend"]].sum()),
        est_pay=("pay", "sum"),
    ).sort_values("hours", ascending=False)
    return report.round(2), len(files)

report, weeks = roster_report("data/rosters")
print(f"Roster report: {weeks} weeks")
print(report)
report.to_csv("roster_report.csv")

### Make it yours
Flag anyone who worked more than 38 hours in a single week.

In [ ]:
import pandas as pd
from pathlib import Path
frames = [pd.read_csv(f).assign(week=int(f.stem.split("_")[-1])) for f in sorted(Path("data/rosters").glob("roster_week_*.csv"))]
roster = pd.concat(frames, ignore_index=True)
weekly = roster.groupby(["name", "week"])["hours"].sum()
print(weekly[weekly > 38] if (weekly > 38).any() else "Nobody worked more than 38 hours in a week.")